In [1]:
from google.colab import files
uploaded = files.upload()

Saving studentVle.csv to studentVle.csv
Saving vle.csv to vle.csv
Saving assessments.csv to assessments.csv
Saving courses.csv to courses.csv
Saving studentAssessment.csv to studentAssessment.csv
Saving studentInfo.csv to studentInfo.csv
Saving studentRegistration.csv to studentRegistration.csv


In [2]:
import pandas as pd

studentVle = pd.read_csv('studentVle.csv')

# Is (id_student, id_site, date) actually unique, or are there real duplicates?
dupe_check = studentVle.groupby(['id_student', 'id_site', 'date']).size()
print(f"Total groups: {len(dupe_check)}, groups with >1 row: {(dupe_check > 1).sum()}")
print(dupe_check[dupe_check > 1].head(10))

Total groups: 8459320, groups with >1 row: 1614505
id_student  id_site  date
6516        877011   204     2
                     209     2
                     218     2
            877012   0       4
                     5       2
                     10      2
                     11      3
                     13      2
                     14      2
                     19      2
dtype: int64


In [4]:
vle = pd.read_csv('vle.csv')
studentVle = pd.read_csv('studentVle.csv')

# dedupe: collapse repeated (student, site, date) rows into one summed row
studentVle_dedup = studentVle.groupby(
    ['code_module', 'code_presentation', 'id_student', 'id_site', 'date'],
    as_index=False
)['sum_click'].sum()

print(f"Before dedup: {len(studentVle)} rows, after dedup: {len(studentVle_dedup)} rows")

merged = studentVle_dedup.merge(
    vle[['id_site', 'code_module', 'code_presentation', 'activity_type']],
    on=['id_site', 'code_module', 'code_presentation'], how='left'
)

click_by_type = merged.groupby('activity_type')['sum_click'].sum().sort_values(ascending=False)
pct = (click_by_type / click_by_type.sum() * 100).round(2)
print(pd.DataFrame({'total_clicks': click_by_type, 'pct_of_all_engagement': pct}))

oue_clicks = merged[merged['activity_type'] == 'ouelluminate'].groupby(
    ['code_module', 'code_presentation'])['sum_click'].sum()
print("\nouelluminate click volume by module/presentation:")
print(oue_clicks if len(oue_clicks) else "No ouelluminate clicks found in the dataset.")

Before dedup: 10655280 rows, after dedup: 8459320 rows
                total_clicks  pct_of_all_engagement
activity_type                                      
oucontent           11206803                  28.30
forumng              7973390                  20.13
quiz                 6981240                  17.63
homepage             6949064                  17.55
subpage              3411582                   8.61
resource             1110132                   2.80
ouwiki                894512                   2.26
url                   566702                   1.43
oucollaborate         108974                   0.28
glossary               87962                   0.22
questionnaire          64764                   0.16
externalquiz           64292                   0.16
page                   63631                   0.16
dataplus               47468                   0.12
ouelluminate           39028                   0.10
dualpane               20716                   0.05
htmlactiv

In [5]:
print(len(studentVle_dedup), len(merged))

8459320 8459320


In [6]:
import pandas as pd

studentAssessment = pd.read_csv('studentAssessment.csv')
assessments = pd.read_csv('assessments.csv')
studentInfo = pd.read_csv('studentInfo.csv')
studentRegistration = pd.read_csv('studentRegistration.csv')

# 1. Does weight sum to 100 within each module-presentation? Check for zero-weight rows
weight_check = assessments.groupby(['code_module','code_presentation'])['weight'].sum()
print("Weight sums per module-presentation (should be ~100):")
print(weight_check.describe())
print(f"\nModule-presentations where weight != 100: {(weight_check != 100).sum()} of {len(weight_check)}")

print("\nAssessment types and zero-weight counts:")
print(assessments.groupby('assessment_type')['weight'].agg(['count', lambda x: (x==0).sum()]))

# 2. Missing/ungraded scores in studentAssessment
print(f"\nTotal studentAssessment rows: {len(studentAssessment)}")
print(f"Rows with missing score: {studentAssessment['score'].isna().sum()}")
print(f"Rows with is_banked == 1 (credited from a prior attempt): {(studentAssessment['is_banked']==1).sum() if 'is_banked' in studentAssessment.columns else 'no is_banked column'}")

# 3. Trajectory feasibility: how many assessments does a typical student have per presentation?
merged_assess = studentAssessment.merge(assessments[['id_assessment','code_module','code_presentation','assessment_type','date','weight']], on='id_assessment')
n_assess_per_student = merged_assess.groupby(['id_student','code_module','code_presentation']).size()
print("\nAssessments per student per presentation (need >=2 for a first-to-last trajectory/gain measure):")
print(n_assess_per_student.describe())
print(f"Students with only 1 assessment: {(n_assess_per_student==1).sum()} of {len(n_assess_per_student)}")

# 4. Withdrawal filtering: students who left before their final assessment's due date
reg = studentRegistration[['id_student','code_module','code_presentation','date_unregistration']]
last_due = merged_assess.groupby(['code_module','code_presentation'])['date'].max().rename('last_assessment_due')
check = reg.merge(last_due, on=['code_module','code_presentation'], how='left')
withdrew_early = check[check['date_unregistration'] < check['last_assessment_due']]
print(f"\nStudents who withdrew before final assessment due date: {len(withdrew_early)} of {len(reg)}")
print(f"Students with no withdrawal date (completed/still active): {reg['date_unregistration'].isna().sum()}")

Weight sums per module-presentation (should be ~100):
count     22.000000
mean     195.454545
std       48.572702
min      100.000000
25%      200.000000
50%      200.000000
75%      200.000000
max      300.000000
Name: weight, dtype: float64

Module-presentations where weight != 100: 19 of 22

Assessment types and zero-weight counts:
                 count  <lambda_0>
assessment_type                   
CMA                 76          46
Exam                24           0
TMA                106          10

Total studentAssessment rows: 173912
Rows with missing score: 173
Rows with is_banked == 1 (credited from a prior attempt): 1909

Assessments per student per presentation (need >=2 for a first-to-last trajectory/gain measure):
count    25843.000000
mean         6.729559
std          3.771381
min          1.000000
25%          4.000000
50%          7.000000
75%         10.000000
max         14.000000
dtype: float64
Students with only 1 assessment: 2520 of 25843

Students who withdrew

In [7]:
# Filter to valid, gradeable, non-banked assessments only
valid = merged_assess[(merged_assess['is_banked'] == 0) & (merged_assess['weight'] > 0)]

# Recount assessments per student under these filters
n_valid = valid.groupby(['id_student','code_module','code_presentation']).size()
print(f"Students with >=2 valid assessments: {(n_valid >= 2).sum()} of {len(n_valid)}")

# First and last valid assessment per student-presentation
valid_sorted = valid.sort_values('date')
first_last = valid_sorted.groupby(['id_student','code_module','code_presentation']).agg(
    first_score=('score','first'), last_score=('score','last'),
    n_assessments=('score','size')
).reset_index()
first_last = first_last[first_last['n_assessments'] >= 2].copy()
first_last['performance_gain'] = first_last['last_score'] - first_last['first_score']

# Exclude early withdrawals from this specific set
reg_small = studentRegistration[['id_student','code_module','code_presentation','date_unregistration']]
final = first_last.merge(reg_small, on=['id_student','code_module','code_presentation'], how='left')
last_due2 = valid.groupby(['code_module','code_presentation'])['date'].max().rename('last_valid_due')
final = final.merge(last_due2, on=['code_module','code_presentation'], how='left')
final = final[~(final['date_unregistration'] < final['last_valid_due'])]

print(f"\nFinal analytic sample size: {len(final)} of 32,593 registered students")
print(final['performance_gain'].describe())

# Which module-presentations have weight==100 (clean) — candidates for a
# weighted-score cross-check / validation subsample, same pattern as the
# ouelluminate robustness check
clean_weight_modules = weight_check[weight_check == 100].index.tolist()
print(f"\nModule-presentations with weight sum == 100: {clean_weight_modules}")

Students with >=2 valid assessments: 20807 of 23239

Final analytic sample size: 18364 of 32,593 registered students
count    18363.000000
mean        -5.560148
std         20.345416
min        -95.000000
25%        -17.000000
50%         -4.000000
75%          7.000000
max         95.000000
Name: performance_gain, dtype: float64

Module-presentations with weight sum == 100: [('GGG', '2013J'), ('GGG', '2014B'), ('GGG', '2014J')]


In [8]:
final = final.dropna(subset=['first_score', 'last_score'])
print(f"After dropping NaN scores: {len(final)}")

After dropping NaN scores: 18363


In [9]:
# What assessment_type is usually the "first" and "last" one?
last_types = valid_sorted.groupby(['id_student','code_module','code_presentation']).agg(
    first_type=('assessment_type','first'), last_type=('assessment_type','last')
)
last_types = last_types.loc[final.set_index(['id_student','code_module','code_presentation']).index.intersection(last_types.index)]
print("Last assessment type distribution:")
print(last_types['last_type'].value_counts())
print("\nFirst assessment type distribution:")
print(last_types['first_type'].value_counts())

# Mean gain broken out by whether the last assessment was an Exam or not
final_with_types = final.merge(last_types.reset_index(), on=['id_student','code_module','code_presentation'])
print("\nMean performance_gain by last-assessment type:")
print(final_with_types.groupby('last_type')['performance_gain'].agg(['mean','std','count']))

Last assessment type distribution:
last_type
TMA     10558
Exam     4937
CMA      2868
Name: count, dtype: int64

First assessment type distribution:
first_type
TMA    15365
CMA     2998
Name: count, dtype: int64

Mean performance_gain by last-assessment type:
                mean        std  count
last_type                             
CMA         8.783821  22.084342   2868
Exam      -11.245088  19.140313   4937
TMA        -6.798257  18.538691  10558


In [10]:
print(final['performance_gain'].quantile([0.01, 0.05, 0.95, 0.99]))

0.01   -61.38
0.05   -42.00
0.95    26.00
0.99    40.00
Name: performance_gain, dtype: float64


In [11]:
lower, upper = final['performance_gain'].quantile([0.01, 0.99])
final['performance_gain_raw'] = final['performance_gain']  # keep original for reference
final['performance_gain'] = final['performance_gain'].clip(lower, upper)

n_capped = ((final['performance_gain_raw'] < lower) | (final['performance_gain_raw'] > upper)).sum()
print(f"Capped {n_capped} of {len(final)} students ({100*n_capped/len(final):.2f}%) at [{lower:.2f}, {upper:.2f}]")
print(final['performance_gain'].describe())

Capped 353 of 18363 students (1.92%) at [-61.38, 40.00]
count    18363.000000
mean        -5.543153
std         19.850827
min        -61.380000
25%        -17.000000
50%         -4.000000
75%          7.000000
max         40.000000
Name: performance_gain, dtype: float64


In [12]:
import pandas as pd

# --- Rebuild merged (in case your session was reset) ---
vle = pd.read_csv('vle.csv')
studentVle = pd.read_csv('studentVle.csv')

studentVle_dedup = studentVle.groupby(
    ['code_module', 'code_presentation', 'id_student', 'id_site', 'date'],
    as_index=False
)['sum_click'].sum()

merged = studentVle_dedup.merge(
    vle[['id_site', 'code_module', 'code_presentation', 'activity_type']],
    on=['id_site', 'code_module', 'code_presentation'], how='left'
)

# --- Build the two engagement measures (this was the missing step) ---
oucontent_engagement = merged[merged['activity_type'] == 'oucontent'].groupby(
    ['id_student', 'code_module', 'code_presentation']
)['sum_click'].sum().reset_index().rename(columns={'sum_click': 'oucontent_clicks'})

ouelluminate_engagement = merged[merged['activity_type'] == 'ouelluminate'].groupby(
    ['id_student', 'code_module', 'code_presentation']
)['sum_click'].sum().reset_index().rename(columns={'sum_click': 'ouelluminate_clicks'})

print(f"Students with any oucontent engagement: {len(oucontent_engagement)}")
print(f"Students with any ouelluminate engagement: {len(ouelluminate_engagement)}")

# --- Step 2: merge outcome onto both treatment corpora ---
full_corpus = final.merge(
    oucontent_engagement, on=['id_student', 'code_module', 'code_presentation'], how='inner'
)
print(f"\nFull corpus (oucontent treatment + valid outcome): {len(full_corpus)} of 18,363")

oue_subsample = final.merge(
    ouelluminate_engagement, on=['id_student', 'code_module', 'code_presentation'], how='inner'
)
print(f"Ouelluminate subsample (multimedia treatment + valid outcome): {len(oue_subsample)}")
print(oue_subsample.groupby(['code_module', 'code_presentation']).size())

# --- Step 3: pull confounders ---
studentInfo = pd.read_csv('studentInfo.csv')
conf_cols = ['id_student', 'code_module', 'code_presentation', 'gender', 'region',
             'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts',
             'studied_credits', 'disability']

full_corpus = full_corpus.merge(studentInfo[conf_cols],
                                  on=['id_student', 'code_module', 'code_presentation'], how='left')
oue_subsample = oue_subsample.merge(studentInfo[conf_cols],
                                      on=['id_student', 'code_module', 'code_presentation'], how='left')

print("\nMissingness in full_corpus confounders (%):")
print((full_corpus[conf_cols].isna().mean() * 100).sort_values(ascending=False))

Students with any oucontent engagement: 26922
Students with any ouelluminate engagement: 2501

Full corpus (oucontent treatment + valid outcome): 17529 of 18,363
Ouelluminate subsample (multimedia treatment + valid outcome): 2031
code_module  code_presentation
BBB          2013B                390
DDD          2013B                649
FFF          2013B                992
dtype: int64

Missingness in full_corpus confounders (%):
imd_band                4.735011
code_module             0.000000
id_student              0.000000
code_presentation       0.000000
gender                  0.000000
region                  0.000000
highest_education       0.000000
age_band                0.000000
num_of_prev_attempts    0.000000
studied_credits         0.000000
disability              0.000000
dtype: float64


In [13]:
print(full_corpus[full_corpus['imd_band'].isna()]['region'].value_counts())
print(full_corpus['region'].value_counts())

region
North Region            547
Ireland                 209
South Region             34
West Midlands Region     28
Scotland                  6
South West Region         3
North Western Region      2
Yorkshire Region          1
Name: count, dtype: int64
region
Scotland                2095
East Anglian Region     1780
South Region            1735
London Region           1564
North Western Region    1386
South West Region       1303
West Midlands Region    1262
Wales                   1206
East Midlands Region    1189
South East Region       1143
North Region            1049
Yorkshire Region        1002
Ireland                  815
Name: count, dtype: int64


In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/CausalMedia-GH', exist_ok=True)

full_corpus.to_csv('/content/drive/MyDrive/CausalMedia-GH/oulad_full_corpus.csv', index=False)
oue_subsample.to_csv('/content/drive/MyDrive/CausalMedia-GH/oulad_ouelluminate_subsample.csv', index=False)

print("Saved to Drive:")
print(f"  Full corpus: {len(full_corpus)} rows")
print(f"  Ouelluminate subsample: {len(oue_subsample)} rows")

Mounted at /content/drive
Saved to Drive:
  Full corpus: 17529 rows
  Ouelluminate subsample: 2031 rows
